# IndoMultiDomain V1 — Final SRC006 Fix

Memaksa SRC006 memakai `reviewContent`, bukan `reviewTitle`, lalu membangun ulang Core.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/IndoMultiDomain')


Mounted at /content/drive


In [2]:
patch_code='\nfrom pathlib import Path\nimport pandas as pd, numpy as np, re, hashlib, unicodedata, json\n\nSEED=20260915\nnp.random.seed(SEED)\n\ndef norm_text(x):\n    if pd.isna(x): return ""\n    x=unicodedata.normalize("NFC",str(x))\n    return re.sub(r"\\s+"," ",x).strip()\n\ndef thash(x):\n    return hashlib.sha256(norm_text(x).encode("utf-8")).hexdigest()\n\ndef split_hash(h):\n    n=int(h[:8],16)%100\n    return "train" if n<80 else ("validation" if n<90 else "test")\n\ndef rebuild_src006(ROOT):\n    candidates=list((ROOT/"02_raw_sources"/"SRC006").rglob("Raw Dataset.csv"))\n    if not candidates:\n        raise RuntimeError("Raw Dataset.csv SRC006 tidak ditemukan")\n    p=candidates[0]\n    df=pd.read_csv(p,low_memory=False)\n    if "reviewContent" not in df.columns:\n        raise RuntimeError(f"reviewContent tidak ditemukan. Kolom: {list(df.columns)}")\n\n    work=df.copy()\n    work["text"]=work["reviewContent"].map(norm_text)\n    work=work[work["text"].str.len()>0].copy()\n    work["duplicate_group"]=work["text"].map(thash)\n    work=work.drop_duplicates("duplicate_group",keep="first").copy()\n\n    print("SRC006 reviewContent nonempty unique:", len(work))\n\n    target=min(15000,len(work))\n    # stratified by category x rating, with deterministic cap\n    strata=["category","rating"]\n    pieces=[]\n    groups=list(work.groupby(strata,dropna=False))\n    for _,g in groups:\n        n=max(1,round(target*len(g)/len(work)))\n        pieces.append(g.sample(min(n,len(g)),random_state=SEED))\n    samp=pd.concat(pieces).drop_duplicates("duplicate_group").copy()\n\n    if len(samp)>target:\n        samp=samp.sample(target,random_state=SEED)\n    elif len(samp)<target:\n        remain=work[~work["duplicate_group"].isin(set(samp["duplicate_group"]))]\n        add=min(target-len(samp),len(remain))\n        if add:\n            samp=pd.concat([samp,remain.sample(add,random_state=SEED)],ignore_index=True)\n\n    samp=samp.reset_index(drop=True)\n\n    out=pd.DataFrame()\n    out["text"]=samp["text"]\n    out["domain"]="ecommerce_retail"\n    out["subdomain"]="product_review"\n    out["genre"]="user_review"\n    out["source_id"]="SRC006"\n    out["source_repository"]="mendeley"\n    out["source_dataset"]="Dataset Customer Review Indonesia"\n    out["source_version"]="1"\n    out["source_platform"]=samp["clientType"].astype(str) if "clientType" in samp.columns else ""\n    out["original_id"]=""\n    out["language"]="id"\n    if "boughtDate" in samp.columns:\n        dt=pd.to_datetime(samp["boughtDate"],errors="coerce",dayfirst=True)\n        out["year"]=dt.dt.year.astype("Int64")\n    else:\n        out["year"]=pd.Series([pd.NA]*len(out),dtype="Int64")\n    out["original_task"]=""\n    out["original_label"]=""\n    out["original_label_type"]=""\n    out["license"]="CC BY 4.0"\n    out["provenance_note"]=f"SRC006; source_file={p}; text_field=reviewContent"\n    out["char_count"]=out["text"].str.len()\n    out["word_count"]=out["text"].str.split().str.len()\n    out["quality_flag"]=np.where(out["char_count"]<10,"short_text","pass")\n    out["duplicate_group"]=out["text"].map(thash)\n    out["split"]=out["duplicate_group"].map(split_hash)\n    out["imd_id"]=[f"IMD-ECO-SRC006-{i+1:08d}" for i in range(len(out))]\n\n    cols=["imd_id","text","domain","subdomain","genre","source_id","source_repository",\n          "source_dataset","source_version","source_platform","original_id","language","year",\n          "original_task","original_label","original_label_type","license","provenance_note",\n          "char_count","word_count","quality_flag","duplicate_group","split"]\n    out=out[cols]\n\n    od=ROOT/"03_harmonized"/"source_level"\n    out.to_parquet(od/"SRC006_harmonized.parquet",index=False)\n    out.to_csv(od/"SRC006_harmonized.csv",index=False)\n    print("SRC006 FINAL:",len(out))\n    return out\n\ndef rebuild_core(ROOT):\n    od=ROOT/"03_harmonized"/"source_level"\n    parts=[]\n    for p in sorted(od.glob("SRC*_harmonized.parquet")):\n        df=pd.read_parquet(p)\n        parts.append(df)\n        print("using",p.name,len(df))\n    core=pd.concat(parts,ignore_index=True)\n    core["duplicate_group"]=core["text"].map(thash)\n    before=len(core)\n    ad=ROOT/"04_analysis"\n    core[core.duplicated("duplicate_group",keep=False)].sort_values("duplicate_group").to_csv(ad/"exact_duplicate_report.csv",index=False)\n    core=core.drop_duplicates("duplicate_group").reset_index(drop=True)\n    core["split"]=core["duplicate_group"].map(split_hash)\n    hd=ROOT/"03_harmonized"\n    core.to_parquet(hd/"indomultidomain_core_v1.parquet",index=False)\n    core.to_csv(hd/"indomultidomain_core_v1.csv",index=False)\n    for s in ["train","validation","test"]:\n        core[core["split"]==s].to_parquet(hd/f"indomultidomain_{s}_v1.parquet",index=False)\n\n    core.groupby("domain").agg(\n        records=("imd_id","count"),words=("word_count","sum"),chars=("char_count","sum"),\n        median_words=("word_count","median"),sources=("source_id","nunique"),genres=("genre","nunique")\n    ).reset_index().to_csv(ad/"domain_statistics.csv",index=False)\n\n    core.groupby(["source_id","source_dataset"]).agg(\n        records=("imd_id","count"),words=("word_count","sum"),median_words=("word_count","median")\n    ).reset_index().to_csv(ad/"source_statistics.csv",index=False)\n\n    core.groupby("genre").agg(\n        records=("imd_id","count"),words=("word_count","sum"),median_words=("word_count","median")\n    ).reset_index().to_csv(ad/"genre_statistics.csv",index=False)\n\n    summary={\n        "records":int(len(core)),\n        "records_before_exact_dedup":int(before),\n        "domains":int(core.domain.nunique()),\n        "genres":int(core.genre.nunique()),\n        "sources":int(core.source_id.nunique()),\n        "words":int(core.word_count.sum())\n    }\n    (ad/"build_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")\n    print(json.dumps(summary,indent=2))\n    return core,summary\n'
(ROOT/'06_builder'/'indomultidomain_fix_src006_reviewcontent.py').write_text(patch_code,encoding='utf-8')
import importlib.util
p=ROOT/'06_builder'/'indomultidomain_fix_src006_reviewcontent.py'
spec=importlib.util.spec_from_file_location('fix3',p)
fix3=importlib.util.module_from_spec(spec)
spec.loader.exec_module(fix3)


In [3]:
src6=fix3.rebuild_src006(ROOT)


SRC006 reviewContent nonempty unique: 38065


/content/drive/MyDrive/IndoMultiDomain/06_builder/indomultidomain_fix_src006_reviewcontent.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt=pd.to_datetime(samp["boughtDate"],errors="coerce",dayfirst=True)


SRC006 FINAL: 15000


In [4]:
core,summary=fix3.rebuild_core(ROOT)


using SRC001_harmonized.parquet 1825
using SRC002_harmonized.parquet 12000
using SRC003_harmonized.parquet 15000
using SRC004_harmonized.parquet 5000
using SRC005_harmonized.parquet 4000
using SRC006_harmonized.parquet 15000
using SRC007_harmonized.parquet 1499
using SRC008_harmonized.parquet 15000
using SRC009_harmonized.parquet 162
{
  "records": 69075,
  "records_before_exact_dedup": 69486,
  "domains": 8,
  "genres": 3,
  "sources": 9,
  "words": 1418001
}


In [5]:
assert core.imd_id.is_unique
assert core.duplicate_group.is_unique
assert core.text.str.len().gt(0).all()
print('SANITY CHECK: PASS')
print(summary)


SANITY CHECK: PASS
{'records': 69075, 'records_before_exact_dedup': 69486, 'domains': 8, 'genres': 3, 'sources': 9, 'words': 1418001}
